# 01 — EDA & Demand Analysis
Ride-hailing marketplace data · Berlin zones

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

In [ ]:
df = pd.read_csv('sample_rides.csv', parse_dates=['timestamp'])
print(f'Rows: {len(df):,} | Cols: {df.shape[1]}')
df.head()

In [ ]:
df.describe().round(3)

## Demand & surge by hour

In [ ]:
hourly = df.groupby('hour').agg(
    demand=('demand_score','mean'),
    supply=('supply_score','mean'),
    surge=('surge_multiplier','mean')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hourly['hour'], hourly['demand'], label='Demand', lw=2, color='#D85A30')
axes[0].plot(hourly['hour'], hourly['supply'], label='Supply',  lw=2, color='#378ADD')
axes[0].set(title='Demand vs Supply by Hour', xlabel='Hour')
axes[0].legend()
axes[1].bar(hourly['hour'], hourly['surge'], color='#EF9F27', alpha=0.85)
axes[1].axhline(1.0, color='gray', ls='--', lw=1)
axes[1].set(title='Avg Surge by Hour', xlabel='Hour')
plt.tight_layout()
plt.show()

## Weekday patterns

In [ ]:
day_map = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}
daily = df.groupby('weekday').agg(
    surge=('surge_multiplier','mean'),
    price=('final_price_eur','mean'),
    conv=('conversion','mean')
).reset_index()
daily['day'] = daily['weekday'].map(day_map)
colors = ['#378ADD']*5 + ['#D85A30']*2

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, title in zip(axes, ['surge','price','conv'], ['Avg Surge','Avg Price (EUR)','Conversion Rate']):
    ax.bar(daily['day'], daily[col], color=colors, alpha=0.85)
    ax.set_title(title)
plt.tight_layout()
plt.show()

## Zone analysis

In [ ]:
zone = df.groupby('zone').agg(
    rides=('conversion','count'),
    surge=('surge_multiplier','mean'),
    conv=('conversion','mean'),
    ds=('demand_supply_ratio','mean')
).sort_values('rides', ascending=False).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].barh(zone['zone'], zone['rides'], color='#378ADD', alpha=0.8)
axes[0].set(title='Rides by Zone', xlabel='Count')
sc = axes[1].scatter(zone['ds'], zone['surge'], s=zone['rides']*5,
    c=zone['conv'], cmap='RdYlGn', alpha=0.85)
plt.colorbar(sc, ax=axes[1], label='Conv rate')
for _, r in zone.iterrows():
    axes[1].annotate(r['zone'], (r['ds'], r['surge']), fontsize=8, ha='center', va='bottom')
axes[1].set(title='D/S Ratio vs Surge', xlabel='D/S Ratio', ylabel='Surge')
plt.tight_layout()
plt.show()
print(zone.round(3).to_string(index=False))

## Price elasticity

In [ ]:
df['price_bin'] = pd.qcut(df['final_price_eur'], q=8, duplicates='drop')
elast = df.groupby('price_bin', observed=True).agg(
    conv=('conversion','mean'),
    price=('final_price_eur','mean')
).reset_index()

plt.figure(figsize=(8, 4))
plt.plot(elast['price'], elast['conv'], marker='o', lw=2, color='#D85A30')
plt.fill_between(elast['price'], elast['conv'], alpha=0.1, color='#D85A30')
plt.title('Price Elasticity of Demand')
plt.xlabel('Avg Final Price (EUR)')
plt.ylabel('Conversion Rate')
plt.tight_layout()
plt.show()

d_conv  = elast['conv'].iloc[-1]  - elast['conv'].iloc[0]
d_price = elast['price'].iloc[-1] - elast['price'].iloc[0]
coef = (d_conv/elast['conv'].mean()) / (d_price/elast['price'].mean())
print(f'Price elasticity coefficient: {coef:.3f}')